# LightGBM

Цель: обучить LightGBM на пяти готовых фолдах и получить вероятности `satisfaction = 1`. Основная метрика — ROC AUC.

## Порядок работы

1. Загрузить `train_with_folds.parquet` и `test_clean.parquet`.
2. Отделить `satisfaction`, `fold` и `id` от признаков.
3. Преобразовать `Class` в pandas `category` с одинаковым списком категорий в train и test.
4. Создать массивы для OOF и усреднённых test-предсказаний.
5. Обучить пять моделей: текущий `fold` использовать для validation, остальные четыре — для обучения.
6. Использовать early stopping и считать ROC AUC каждого фолда.
7. Записать validation-вероятности в OOF, а test-вероятности усреднить по пяти моделям.

Начальные настройки: `n_estimators=2000`, `learning_rate=0.03`, `num_leaves=31`, `subsample=0.8`, `subsample_freq=1`, `colsample_bytree=0.8`, `reg_lambda=1.0`, `random_state=42`.

При обучении использовать `eval_metric="auc"` и callbacks `early_stopping(100)`, `log_evaluation(100)`. Не создавать новый `train_test_split`, не масштабировать признаки и не заполнять числовые пропуски. Использовать `predict_proba(data)[:, 1]`.

## Результаты

Сохранить:

- `artifacts/oof/lightgbm.npy`;
- `artifacts/test_predictions/lightgbm.npy`;
- `artifacts/metrics/lightgbm.json` с параметрами, fold AUC, OOF AUC и лучшими итерациями.

Проверить длины массивов, диапазон вероятностей от 0 до 1, тип `category` у `Class` и отсутствие `id`, `fold`, `satisfaction` среди признаков.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OOF_DIR = PROJECT_ROOT / "artifacts" / "oof"
TEST_PREDICTIONS_DIR = PROJECT_ROOT / "artifacts" / "test_predictions"
METRICS_DIR = PROJECT_ROOT / "artifacts" / "metrics"

TRAIN_PATH = PROCESSED_DATA_DIR / "train_with_folds.parquet"
TEST_PATH = PROCESSED_DATA_DIR / "test_clean.parquet"
SAMPLE_SUBMISSION_PATH = RAW_DATA_DIR / "sample_submission.csv"

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)

train_df.shape, test_df.shape, sample_submission_df.shape